# K2 (mock feed) — the update path against a server of its own

**What this is:** the app's own update path (check the feed, download, signature check, swap and restart, tampered and broken
downloads) against a **mock GitHub on 127.0.0.1**, in a **data folder of its own**: nothing touches your installed app, its
config, or the internet. It needs an exe built with the test flags (`--features test-env`, see the README).

**No clicks:** the pipeline is Python (`runbook/pipelines/updater_mock.py`); the app is driven over the **bridge** (MQTT): the
notebook sends `start-update` / `restart-update` (the dialog's buttons) and waits on the app's `update-state` and `app-started`
events. What the dialog *looked like* is not checked here (rows `skip`; set `UI_CHECKS = True` below to be asked).
The same pipeline runs without Jupyter: `python -m runbook.run updater-mock --exe <exe> --key <backup key>`.

**You type** the exe path and the BACKUP key path, and the key's password **once per exe** (the signature is kept in
`runs/mock/signed/`). Set `RUNBOOK_TEXT_LOCAL_EXE`, `RUNBOOK_TEXT_BACKUP_KEY` and `RUNBOOK_TEXT_BACKUP_PW` to be asked nothing.
Needs Node 20+ for the bridge. Run Jupyter as Administrator and close the installed app and any other copy first.

In [ ]:
import sys
from pathlib import Path
here = Path.cwd().resolve()              # Jupyter starts a notebook in its own folder:
root = next(p for p in [here, *here.parents] if (p / "runbook" / "common.py").exists())
sys.path.insert(0, str(root))            # ...so find runbook above it
from runbook import bridge, common
from runbook.common import Recorder, ask_text
from runbook.pipelines.updater_mock import UpdaterMock
import os
rec = Recorder("updater-mock")
if not common.require_windows_admin(rec):
    raise SystemExit("Start Jupyter from an Administrator terminal, then run again.")
bridge.ensure_installed()                # npm ci in runbook/bridge, the first time

## The exe, the key, and the run

In [ ]:
UI_CHECKS = False
EXE = Path(ask_text("LOCAL_EXE", "Full path of the test-env exe (e.g. C:/.../target/release/resonance-stream.exe), then Enter: "))
if not EXE.is_file():
    raise SystemExit(f"{EXE} is not a file")
KEY = ask_text("BACKUP_KEY", "Full path of the BACKUP key file (e.g. C:/keys/backup.key), then Enter: ")
pipeline = UpdaterMock(rec, EXE, common.RUNS, ui_checks=UI_CHECKS)
pipeline.run(key_path=KEY, password=os.environ.get("RUNBOOK_TEXT_BACKUP_PW"))

## Report
Paste the output of the next cell to Claude. The copies, their logs and what the app said on the bridge stay under `runs/mock/` (ignored by git).

In [ ]:
print(rec.report())